# Load a small MFCAD subset

Load a few samples from the published train split and inspect their DGL graph tensors and per-face labels. This notebook expects the dataset to already be unpacked under `project/data/MFCADDataset`.

In [1]:
from collections import Counter

import pandas as pd

from uvnet_project.load_mfcad import DATA_DIR, load_mfcad

In [2]:
print(f"Using MFCAD at: {DATA_DIR}")

Using MFCAD at: C:\Users\KaiSheng\OneDrive\Desktop\GNN-IN-CAD\data\MFCADDataset


In [3]:
samples = load_mfcad("train", head=5)

In [4]:
print(f"Loaded {len(samples)} training samples")

Loaded 5 training samples


## What is one `sample`?

`samples` is a list of CAD solids from the selected split. Each item is a Python dictionary describing **one solid**, not one face. In the summary loop below, `sample` takes each item in turn; after the loop it refers to the **last** loaded solid. Here we explicitly select `samples[0]` to inspect the first solid.

| Key | Python object | Meaning |
| --- | --- | --- |
| `id` | String | Sample ID used to locate the graph and annotation files. |
| `graph` | DGL graph | Nodes are B-rep faces; stored edges connect faces along boundary curves. Geometry is attached as tensors. |
| `faces` | List of dictionaries | One annotation per graph node, including its bounding box and machining-feature class. `faces[i]` describes node `i`. |
| `graph_metadata` | Dictionary | File-level tensor metadata returned by DGL's graph loader; it may be empty. It is separate from the face annotations. |

The loader checks that the number of nodes equals the number of face annotations. It returns the stored graph directly; it does not insert class labels into `graph.ndata`.


In [6]:
example_sample = samples[0]
example_graph = example_sample["graph"]

print("Sample ID:", example_sample["id"])
print("Dictionary keys:", list(example_sample))
print("Faces / nodes:", len(example_sample["faces"]), "/", example_graph.num_nodes())
print("Stored graph edges:", example_graph.num_edges())
print("Node tensor keys:", list(example_graph.ndata))
print("Edge tensor keys:", list(example_graph.edata))
print("Graph-file metadata:", example_sample["graph_metadata"])

pd.DataFrame([
    {"field": key, "type": type(value).__name__,
     "size": len(value) if isinstance(value, (list, dict)) else None}
    for key, value in example_sample.items()
])


Sample ID: 0-0-10-12-19_color
Dictionary keys: ['id', 'graph', 'faces', 'graph_metadata']
Faces / nodes: 17 / 17
Stored graph edges: 84
Node tensor keys: ['x']
Edge tensor keys: ['x']
Graph-file metadata: {}


,field,type,size
0,id,str,NaN
1,graph,DGLGraph,NaN
2,faces,list,17.0
3,graph_metadata,dict,0.0


## How faces, edges and annotations line up

For a solid with `N` faces and `E` stored graph edges:

| Expression | Shape / contents | Interpretation |
| --- | --- | --- |
| `graph.ndata["x"]` | `(N, 10, 10, 7)` | One 10 by 10 surface UV grid per face. Channels 0-2: xyz position; 3-5: surface normal xyz; 6: binary trimming mask (1 = inside the trimmed face). |
| `graph.ndata["x"][i]` | `(10, 10, 7)` | Surface samples for graph face `i`. These dimensions are UV samples, not an RGB image. |
| `graph.edata["x"]` | `(E, 10, 6)` | Ten curve samples per stored edge. Channels 0-2: xyz position; 3-5: curve tangent xyz. |
| `graph.edges(order="eid")` | Source and destination node-index tensors | Entry `e` gives the endpoint faces for edge-geometry row `e`. |
| `sample["faces"][i]["segment"]` | `name` and `index` | Ground-truth machining-feature class for face `i`. The numeric index is the classification target. |
| `sample["faces"][i]["bbox"]` | xmin, ymin, zmin, xmax, ymax, zmax | Axis-aligned face bounding box; this is not its surface area. |

Node indices are zero-based graph/annotation positions, not verified STEP-viewer face IDs. Stored graph edges may contain both directions and multiple boundary curves between the same pair of faces, so `E` is not necessarily the number of unique undirected face adjacencies.

The following cell inspects face 0 and the first few edges. Labels come from the annotation list, rather than an assumed `graph.ndata["y"]` field. These are input features and ground-truth labels, not model predictions.


In [7]:
face_index = 0
face_annotation = example_sample["faces"][face_index]
face_grid = example_graph.ndata["x"][face_index]

print("Face index:", face_index)
print("Class:", face_annotation["segment"])
print("Bounding box:", face_annotation["bbox"])
print("Face grid shape:", tuple(face_grid.shape))
print("One UV sample [x, y, z, nx, ny, nz, mask]:", face_grid[0, 0].tolist())

source_faces, destination_faces = example_graph.edges(order="eid")
edge_count = min(5, example_graph.num_edges())
edge_preview = pd.DataFrame({
    "edge_index": range(edge_count),
    "source_face": source_faces[:edge_count].tolist(),
    "destination_face": destination_faces[:edge_count].tolist(),
})
print("All face class indices:", [face["segment"]["index"] for face in example_sample["faces"]])
edge_preview


Face index: 0
Class: {'name': 'stock', 'index': 15}
Bounding box: {'xmin': -1.0, 'ymin': -0.5325959886792, 'zmin': -1.0, 'xmax': -1.0, 'ymax': 1.0, 'zmax': 1.0}
Face grid shape: (10, 10, 7)
One UV sample [x, y, z, nx, ny, nz, mask]: [-1.0, 1.0, 1.0, -1.0, -0.0, -0.0, 1.0]
All face class indices: [15, 15, 0, 15, 15, 0, 15, 12, 12, 12, 15, 12, 10, 10, 10, 10, 10]


,edge_index,source_face,destination_face
0,0,0,1
1,1,0,3
2,2,0,5
3,3,0,4
4,4,0,2


In [5]:
# List all dataset classes, including stock/background, beyond the five samples.
import json

class_pairs = set()
for label_path in sorted((DATA_DIR / "labels").glob("*_ids.json")):
    with label_path.open() as label_file:
        faces = json.load(label_file)["body"]["faces"]
    class_pairs.update(
        (face["segment"]["index"], face["segment"]["name"])
        for face in faces
    )

machining_classes = pd.DataFrame(
    sorted(class_pairs), columns=["class_index", "class_name"]
)
print(f"Total classes (including stock): {len(machining_classes)}")
machining_classes


Total classes (including stock): 16


,class_index,class_name
0,0,rectangular_through_slot
1,1,triangular_through_slot
2,2,rectangular_passage
3,3,triangular_passage
4,4,6sides_passage
5,5,rectangular_through_step
6,6,2sides_through_step
7,7,slanted_through_step
8,8,rectangular_blind_step
9,9,triangular_blind_step


In [8]:
summary = []
for sample in samples:
    graph = sample["graph"]
    feature_names = [face["segment"]["name"] for face in sample["faces"]]
    summary.append({
        "sample_id": sample["id"],
        "nodes/faces": graph.num_nodes(),
        "edges": graph.num_edges(),
        "node_x_shape": tuple(graph.ndata["x"].shape),
        "edge_x_shape": tuple(graph.edata["x"].shape),
        "face classes": dict(Counter(feature_names)),
    })

pd.DataFrame(summary)

,sample_id,nodes/faces,edges,node_x_shape,edge_x_shape,face classes
0,0-0-10-12-19_color,17,84,"(17, 10, 10, 7)","(84, 10, 6)","{'stock': 6, 'rectangular_through_slot': 2, 't..."
1,1-7-7-8-13-23_color,19,102,"(19, 10, 10, 7)","(102, 10, 6)","{'stock': 6, 'rectangular_blind_step': 2, 'sla..."
2,3-3-4-5-5-23_color,29,162,"(29, 10, 10, 7)","(162, 10, 6)","{'stock': 9, '6sides_passage': 2, 'rectangular..."
3,1-1-5-6-13-23_color,20,108,"(20, 10, 10, 7)","(108, 10, 6)","{'stock': 7, 'rectangular_through_step': 3, '2..."
4,1-3-5-8-14-23_color,24,132,"(24, 10, 10, 7)","(132, 10, 6)","{'stock': 7, 'rectangular_blind_step': 2, 'rec..."


In [9]:
# Inspect the first graph and its first few face annotations.
first = samples[0]
print(first["graph"])
pd.json_normalize(first["faces"]).head()

Graph(num_nodes=17, num_edges=84,
      ndata_schemes={'x': Scheme(shape=(10, 10, 7), dtype=torch.float32)}
      edata_schemes={'x': Scheme(shape=(10, 6), dtype=torch.float32)})


,bbox.xmin,bbox.ymin,bbox.zmin,bbox.xmax,bbox.ymax,bbox.zmax,segment.name,segment.index
0,-1.0,-0.532596,-1.000000,-1.000000,1.000000,1.0,stock,15
1,-1.0,-1.000000,1.000000,1.000000,1.000000,1.0,stock,15
2,-1.0,-1.000000,-1.000000,-0.532596,-0.532596,1.0,rectangular_through_slot,0
3,-1.0,1.000000,-0.771569,1.000000,1.000000,1.0,stock,15
4,-1.0,-1.000000,-1.000000,1.000000,0.771569,-1.0,stock,15
